In [11]:
# Hamza Faheem DS-B,  AI Assignment #2

In [18]:
import random
import copy

In [19]:
class Card:
    def __init__(self,color,val):
     self.color = color
     self.val = val
        
    def __repr__(self):
     return ((self.color) + " " + str(self.val))
        
    def __eq__(self,other):
     if not isinstance(other, Card): 
       return False
     return self.color == other.color and self.val == other.val

def generateDeck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    values = [str(i) for i in range(10)] + ['Skip']

    deck = []
    for c in colors:
        for v in values:
            new_card = Card(c, v)
            deck.append(new_card)
            
    random.shuffle(deck)
    return deck        

def getValidMoves(hand, topC):    
    validMs = []
    for card in hand:
        if card.color == topC.color or card.val == topC.val:
            validMs.append(card)
    return validMs

In [20]:
class gameState:
    def __init__(self, p1c, p2c, p3c, deck, top, currTurn = 1):
      self.units = {1 : p1c, 2 : p2c, 3 : p3c}  
      self.top = top
      self.deck = deck
      self.currTurn = currTurn

    def clone(self):
      return copy.deepcopy(self)

    def apply(self, P_id, cardsPlayed = None):
      nextTurn = (P_id % 3) + 1
      if cardsPlayed:
        self.units[P_id].remove(cardsPlayed)
        self.deck.append(self.top) 
        self.top = cardsPlayed
            
        if cardsPlayed.val == 'Skip':
            nextTurn = (nextTurn % 3) + 1
      else:
        if self.deck:
            drawnC = self.deck.pop(0)
            self.units[P_id].append(drawnC)
                
      self.currTurn = nextTurn
      return nextTurn

In [21]:
def evaluate(state, P_id, strat = "baseline"):
    myHands = state.units[P_id]
    oppsHands = [state.units[p] for p in [1, 2, 3] if p != P_id]
    
    cardsOfAI = len(myHands)
    avgOppsCards = sum(len(hand) for hand in oppsHands) / 2
    skipz = sum(1 for card in myHands if card.val == 'Skip')
    
    if cardsOfAI == 0: return 1000  
    if any(len(h) == 0 for h in oppsHands): return -1000 
    
    if strat == "defensive":
        score = 50 - 4 * cardsOfAI + 5 * avgOppsCards + 6 * skipz
    elif strat == "offensive":
        score = 50 - 8 * cardsOfAI + 2 * avgOppsCards + 2 * skipz
    else:
        score = 50 - 5 * cardsOfAI + 2 * avgOppsCards + 3 * skipz
        
    return score

In [22]:
def minimax(state, depth, maximizing_PID):
    if depth == 0 or len(state.units[1]) == 0 or len(state.units[2]) == 0 or len(state.units[3]) == 0:
        return evaluate(state, maximizing_PID, "defensive"), None

    curr_P = state.currTurn
    valid_moves = getValidMoves(state.units[curr_P], state.top)
    
    if len(valid_moves) == 0:
        possibleMs = [None]
    else:
        possibleMs = valid_moves

    if curr_P == maximizing_PID:
        max_eval = float('-inf')
        reasonableMove = None
        
        for move in possibleMs:
            simulated_state = state.clone()
            simulated_state.apply(curr_P, move)
            evaluatedScore, _ = minimax(simulated_state, depth - 1, maximizing_PID)
            
            if evaluatedScore > max_eval:
                max_eval = evaluatedScore
                reasonableMove = move
                
        return max_eval, reasonableMove

    else:
        min_eval = float('inf')
        reasonableMove = None
        
        for move in possibleMs:
            simulated_state = state.clone()
            simulated_state.apply(curr_P, move)
            evaluatedScore, _ = minimax(simulated_state, depth - 1, maximizing_PID)
            
            if evaluatedScore < min_eval:
                min_eval = evaluatedScore
                reasonableMove = move
                
        return min_eval, reasonableMove

In [23]:
def expectimax(state, depth, AI_iden):
    if depth == 0 or len(state.units[1]) == 0 or len(state.units[2]) == 0 or len(state.units[3]) == 0:
        return evaluate(state, AI_iden, "defensive"), None

    currP = state.currTurn
    valid_moves = getValidMoves(state.units[currP], state.top)
    if currP == AI_iden:
        if len(valid_moves) > 0:
            max_eval = float('-inf')
            reasonableMove = None
            
            for move in valid_moves:
                sim_state = state.clone()
                sim_state.apply(currP, move)
                evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
                
                if evaluatedScore > max_eval:
                    max_eval = evaluatedScore
                    reasonableMove = move
            return max_eval, reasonableMove
            
        else:
            if len(state.deck) == 0:
                return evaluate(state, AI_iden, "offensive"), None
                
            expectedVal = 0
            prob = 1.0 / len(state.deck)
            
            for i in range(len(state.deck)):
                sim_state = state.clone()
                drawn_card = sim_state.deck.pop(i)
                sim_state.units[currP].append(drawn_card)
                sim_state.currTurn = (currP % 3) + 1 
                
                evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
                expectedVal += prob * evaluatedScore
                
            return expectedVal, None 

    else:
        if len(valid_moves) == 0:
            possibleMs = [None]
        else:
            possibleMs = valid_moves
            
        expectedVal = 0
        prob = 1.0 / len(possibleMs)
        
        for move in possibleMs:
            sim_state = state.clone()
            sim_state.apply(currP, move)
            evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
            expectedVal += prob * evaluatedScore
            
        return expectedVal, random.choice(possibleMs)    

In [24]:
import time

def print_ai_decisions(state, player_id, valid_moves, algorithm_name):
    print(f"Top card: {state.top}")
    print("AI hand:")
    for c in state.units[player_id]:
        print(c)
        
    print("\nAI decision(All possible decisions considered at depth 1):")
    reasonableMove = None
    bestSCORE = float('-inf')
    
    if len(valid_moves) == 0:
        print("No valid moves. Must Draw.")
        return None
        
    for move in valid_moves:
        sim_state = state.clone()
        sim_state.apply(player_id, move)
        
        if algorithm_name == "minimax":
            score, _ = minimax(sim_state, depth = 2, maximizing_PID = player_id)
        else:
            score, _ = expectimax(sim_state, depth = 2, AI_iden = player_id)
            
        print(f"Play: {move}")
        print(f"Expected score: {score:.1f}")
        
        if score > bestSCORE:
            bestSCORE = score
            reasonableMove = move
            
    print(f"\n--> AI chooses to play: {reasonableMove}\n")
    return reasonableMove

def play_game():
    print("<<< WELCOME TO AI UNO >>>")
    mode = input("Choose mode for USER (Player 3) ==>\nType '1' for Manual Mode\nType '2' for AI Simulation Mode\nChoice = ")

    deck = generateDeck()
    p1_hand = [deck.pop(0) for _ in range(5)]
    p2_hand = [deck.pop(0) for _ in range(5)]
    p3_hand = [deck.pop(0) for _ in range(5)]
    
    topC = deck.pop(0)
    while topC.val == 'Skip':
        deck.append(topC)
        topC = deck.pop(0)
        
    state = gameState(p1_hand, p2_hand, p3_hand, deck, topC)
    
    while True:
        curr = state.currTurn
        print(" ")
        print(f"PLAYER {curr}'S TURN")
        print("*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*")
        
        if len(state.units[1]) == 0:
            print("Player 1 (Defensive AI) Wins!")
            break
        elif len(state.units[2]) == 0:
            print("Player 2 (Offensive AI) Wins!")
            break
        elif len(state.units[3]) == 0:
            print("Player 3 Wins!")
            break
            
        valid_moves = getValidMoves(state.units[curr], state.top)
        chosen_move = None
        
        # *** P1 (Minimax) ***
        if curr == 1:
            print("Player 1 is thinking (Minimax)...")
            chosen_move = print_ai_decisions(state, curr, valid_moves, "minimax")
            
        # *** P2 (Expectimax) ***
        elif curr == 2:
            print("Player 2 is thinking (Expectimax)...")
            chosen_move = print_ai_decisions(state, curr, valid_moves, "expectimax")
            
        # *** P3 (User) ***
        elif curr == 3:
            if mode == '2': 
                print("Player 3 is thinking (Minimax)...")
                chosen_move = print_ai_decisions(state, curr, valid_moves, "minimax")
            else: # MANUAL WORKING
                print(f"Top card: {state.top}")
                print(f"Your hand: {state.units[curr]}")
                if len(valid_moves) == 0:
                    print("You have no valid moves. Drawing a card...")
                    chosen_move = None
                else:
                    print(f"Valid moves: {valid_moves}")
                    choice = input("Enter the index of the card to play (0, 1, 2...) or press Enter to draw: ")
                    if choice.isdigit() and int(choice) < len(valid_moves):
                        chosen_move = valid_moves[int(choice)]
                    else:
                        print("Invalid choice or chose to draw.")
                        chosen_move = None
        
        state.apply(curr, chosen_move)
        time.sleep(1) 

play_game()

<<< WELCOME TO AI UNO >>>


Choose mode for USER (Player 3) ==>
Type '1' for Manual Mode
Type '2' for AI Simulation Mode
Choice =  2


 
PLAYER 1'S TURN
*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*
Player 1 is thinking (Minimax)...
Top card: Yellow 0
AI hand:
Green 7
Blue Skip
Red 1
Blue 6
Green 4

AI decision(All possible decisions considered at depth 1):
No valid moves. Must Draw.
 
PLAYER 2'S TURN
*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*
Player 2 is thinking (Expectimax)...
Top card: Yellow 0
AI hand:
Red 2
Green 3
Yellow Skip
Red 4
Blue 4

AI decision(All possible decisions considered at depth 1):
Play: Yellow Skip
Expected score: 56.5

--> AI chooses to play: Yellow Skip

 
PLAYER 1'S TURN
*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*
Player 1 is thinking (Minimax)...
Top card: Yellow Skip
AI hand:
Green 7
Blue Skip
Red 1
Blue 6
Green 4
Red 6

AI decision(All possible decisions considered at depth 1):
Play: Blue Skip
Expected score: 54.0

--> AI chooses to play: Blue Skip

 
PLAYER 3'S TURN
*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*
Player 3 is thinking (Minimax)...
Top ca

In [25]:
#### GAME SIMULATION USING TKINTER GUI ####

import tkinter as tk
UNO_COLORS = {'Red': '#FF3333', 'Blue': '#3366FF', 'Green': '#33CC33','Yellow': '#FFCC00'}

game = tk.Tk()
game.title("AI UNO GAME")
game.geometry("850x650")
game.configure(bg = "#2b2b2b")

top_frame = tk.Frame(game, bg = "#2b2b2b")
top_frame.pack(side = tk.TOP, pady = 10)

middle_frame = tk.Frame(game, bg = "#2b2b2b")
middle_frame.pack(side = tk.TOP, expand = True, fill = tk.BOTH, padx = 20)

p1_frame = tk.Frame(middle_frame, bg = "#2b2b2b")
p1_frame.pack(side = tk.LEFT, expand = True)

center_frame = tk.Frame(middle_frame, bg = "#2b2b2b")
center_frame.pack(side = tk.LEFT, expand = True)

p3_frame = tk.Frame(middle_frame, bg = "#2b2b2b")
p3_frame.pack(side = tk.RIGHT, expand = True)

logB = tk.Text(game, height = 10, width = 80, font = ("Consolas", 10), bg = "#1e1e1e", fg = "white")
logB.pack(side = tk.BOTTOM, pady = 10)

def log(message):
    logB.insert(tk.END, message + "\n")
    logB.see(tk.END)


def draw_hand(frame, p_id, title_text):
    for widget in frame.winfo_children():
        widget.destroy()
        
    if gs.currTurn == p_id:
        title_bg = "#FFD700"  
        title_fg = "black"    
    else:
        title_bg = "#2b2b2b"  
        title_fg = "white"   
        
    title = tk.Label(frame, text = title_text, font = ("Verdana", 12, "bold"), bg = title_bg, fg = title_fg)
    title.pack(pady = 5)
    
    cards_frame = tk.Frame(frame, bg = "#2b2b2b")
    cards_frame.pack()
    
    for card in gs.units[p_id]:
        bg_color = UNO_COLORS.get(card.color, "white")
        text_color = "black" if card.color == 'Yellow' else "white"
        
        lbl = tk.Label(cards_frame, text = f"{card.val}", bg = bg_color, fg = text_color, width = 4, height = 4, font = ("Arial", 12, "bold"), relief = "raised")
        lbl.pack(side = tk.LEFT, padx = 2)

def update_ui():
    draw_hand(p1_frame, 1, "Player 1 (Defensive)")
    draw_hand(top_frame, 2, "Player 2 (Offensive)")
    draw_hand(p3_frame, 3, "Player 3 (Minimax)")
    
    tc = gs.top
    tc_bg = UNO_COLORS.get(tc.color, "white")
    tc_fg = "black" if tc.color == 'Yellow' else "white"
    
    topCardDisplay.config(text = f"{tc.val}", bg = tc_bg, fg = tc_fg)

deck = generateDeck()
p1_hand = [deck.pop(0) for _ in range(5)]
p2_hand = [deck.pop(0) for _ in range(5)]
p3_hand = [deck.pop(0) for _ in range(5)]

topC = deck.pop(0)
while topC.val == 'Skip':
    deck.append(topC)
    topC = deck.pop(0)

gs = gameState(p1_hand, p2_hand, p3_hand, deck, topC)
tk.Label(center_frame, text = "TOP CARD", font = ("Verdana", 14, "bold"), bg = "#2b2b2b", fg = "white").pack()
topCardDisplay = tk.Label(center_frame, text = "", width = 6, height = 6, font = ("Arial", 20, "bold"), relief = "ridge")
topCardDisplay.pack(pady = 10)

def doNextT():
    for p in [1, 2, 3]:
        if len(gs.units[p]) == 0:
            log(f"\n*~*~*~*~ PLAYER {p} WINS! ~*~*~*~*\n:D")
            next_btn.config(state = tk.DISABLED, text = "GAME OVER")
            update_ui()
            return

    curr = gs.currTurn
    log(f"Player {curr}'s Turn!")
    valid_moves = getValidMoves(gs.units[curr], gs.top)
    reasonableMove = None
    bestSCORE = float('-inf')

    if len(valid_moves) > 0:
        for move in valid_moves:
            sim = gs.clone()
            sim.apply(curr, move)
            
            if curr == 2:
                score, _ = expectimax(sim, depth = 2, AI_iden = curr)
            else:
                score, _ = minimax(sim, depth = 2, maximizing_PID = curr)

            if score > bestSCORE:
                bestSCORE = score
                reasonableMove = move

    if reasonableMove:
        log(f"-> Plays = {reasonableMove.color} {reasonableMove.val}")
    else:
        log("-> No valid moves. Draws a card.")
        
    gs.apply(curr, reasonableMove)
    update_ui() 

next_btn = tk.Button(center_frame, text = "Next Turn ->", font = ("Verdana", 14, "bold"), bg = "#4CAF50", fg = "white", command =  doNextT)
next_btn.pack(pady = 20)

log("Game ready. Click 'Next Turn' to start.")
update_ui()
game.mainloop()